# 02 · MathKG Dataset Builder

Build a real mathematical knowledge graph from **ProofWiki** data,
structured for the NRO pipeline.

This notebook is a thin orchestration wrapper.  All scraping / parsing /
filtering logic lives in `mre.knowledge_graph.mathkg_builder`.

## Source priority
1. Local XML dump (`/kaggle/input/proofwiki-dump/latest.xml.gz`)
2. Auto-download XML dump from `proofwiki.org`
3. MediaWiki API (correct category names: `Category:Proven Results` etc.)
4. Synthetic fallback — always works offline

## Output
```
data/mathkg/
  entities.tsv          entity_id, name, type, description
  relations.tsv         head_id, relation, tail_id, confidence, source_url
  entity_texts.tsv      entity_id, raw_text
  entity_embeddings.npy  SBERT 384-dim (if sentence-transformers installed)
  splits/{rel}_{split}.tsv
  stats.json
```

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from mre.utils import load_config, set_seed, get_logger
from mre.knowledge_graph.mathkg_builder import MathKGBuilder
from mre.knowledge_graph.mathkg_loader  import MathKGLoader

cfg    = load_config()
set_seed(cfg.experiment.seed)
logger = get_logger('notebook_02')

print(f'MAX_ENTITIES  : {cfg.knowledge_graph.max_entities}')
print(f'Output dir    : {cfg.knowledge_graph.data_dir}')

## 1 · Crawl Entities

In [ ]:
builder = MathKGBuilder(
    data_dir    = cfg.knowledge_graph.data_dir,
    max_entities = cfg.knowledge_graph.max_entities,
    seed        = cfg.experiment.seed,
)
builder.crawl_entities()
print(f'Entities collected: {len(builder.entities):,}')

## 2 · Extract Relations

In [ ]:
builder.extract_relations()
print(f'Raw triples: {len(builder.raw_triples):,}')

## 3 · Filter & Split

In [ ]:
builder.filter_and_clean(
    conf_threshold = cfg.knowledge_graph.confidence_threshold,
    min_degree     = cfg.knowledge_graph.min_entity_degree,
)
builder.make_splits()
print(builder.summary())

## 4 · Save & Verify

In [ ]:
builder.save()

# Verify the saved dataset loads correctly
kg = MathKGLoader(cfg.knowledge_graph.data_dir)
print(kg.summary())

print('\nUsable relations:')
for rel in kg.usable_relations(min_train=4):
    train, val, test = kg.get_split(rel)
    print(f'  {rel:20s}  train={len(train):4d}  val={len(val):4d}  test={len(test):4d}')

## 5 · Dataset Statistics

In [ ]:
from mre.knowledge_graph.mathkg_builder import plot_dataset_stats
import matplotlib.pyplot as plt

fig = plot_dataset_stats(builder)
fig.savefig(f'{cfg.knowledge_graph.data_dir}/mathkg_stats.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()